# STEP 4 — Learned Router

**Prerequisite:** Steps 0–3 complete. `results/baseline_steps.csv` must exist.

**What it does:**
1. Loads per-step logit features from Step 2 baselines
2. Trains a logistic regression to predict "should this step escalate?"
3. Runs the learned router on both datasets with a threshold sweep
4. Saves learned router results alongside routing_episodes.csv

**Outputs:**
- `results/learned_router_episodes.csv`
- `results/learned_router_auc.txt`
- `results/learned_router_coefs.json`

**Estimated run time:** ~1 hour (re-runs episodes with the trained router)

**No GPU needed for training** — logistic regression is CPU-only (seconds).

In [ ]:
import sys, gc, json
import torch
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, '/kaggle/working/slm_agents_paper')
from src.session_init import init
P = init()

from src.routing  import train_learned_router, LearnedRouter, LEARNED_THRESHOLDS, LOGIT_FEATURES
from src.models   import load_model, get_model_by_name
from src.datasets import load_samples
from src.react_env import run_react_episode, episode_to_row
from src.metrics   import compute_summary

hotpotqa = load_samples(P['data'] / 'hotpotqa.json')
musique  = load_samples(P['data'] / 'musique.json')

In [ ]:
# Load step-level data from always_slm_1b baseline runs
steps_df = pd.read_csv(P['results'] / 'baseline_steps.csv')
steps_1b = steps_df[steps_df['slm_name'] == 'Qwen2.5-1.5B']
print(f'Training data: {len(steps_1b)} steps from always_slm_1b baseline')
print(f'Episode success rate: {steps_1b["episode_success"].mean():.3f}')

In [ ]:
# Train learned router
learned_router, auc, coefs = train_learned_router(steps_1b, verbose=True)

# Save metadata
(P['results'] / 'learned_router_auc.txt').write_text(f'AUC={auc:.4f}\n')
with open(P['results'] / 'learned_router_coefs.json', 'w') as f:
    json.dump({'auc': auc, 'coefs': {k: float(v) for k, v in coefs.items()}}, f, indent=2)

print(f'\nAUC = {auc:.3f}')
if auc < 0.60:
    print('⚠️  WARNING: AUC < 0.60 — learned router may not be reliable')
    print('   Fallback: report as a negative result in the paper')
else:
    print('✓ Learned router is predictive — include in paper')

In [ ]:
# Run learned router with threshold sweep
LR_PATH = P['results'] / 'learned_router_episodes.csv'
if LR_PATH.exists():
    lr_df = pd.read_csv(LR_PATH)
    done_keys = set(zip(lr_df['policy'], lr_df['slm_name'], lr_df['dataset']))
else:
    lr_df = pd.DataFrame()
    done_keys = set()

all_rows = lr_df.to_dict('records') if not lr_df.empty else []

slm_cfg = get_model_by_name('Qwen2.5-1.5B')
llm_cfg = get_model_by_name('Llama-3.1-8B')
slm_m, slm_t = load_model(slm_cfg)
llm_m, llm_t = load_model(llm_cfg)
print('Models loaded')

In [ ]:
# Custom episode runner that uses learned router's feature-based decision
from src.inference import generate_single_with_logits
from src.react_env import (
    build_react_prompt, parse_react_action,
    _execute_search, _is_finish_action, _extract_finish_answer, _exact_match
)
from src.datasets import build_bm25_index

for T in LEARNED_THRESHOLDS:
    learned_router.set_threshold(T)
    policy_id = f'learned_T{T:.1f}'
    for ds_name, samples in [('hotpotqa', hotpotqa), ('musique', musique)]:
        key = (policy_id, slm_cfg.name, ds_name)
        if key in done_keys:
            continue
        print(f'  [{policy_id}] {ds_name}...')
        for sample in samples:
            question = sample['question']
            gold     = sample['answer']
            passages = sample['passages']
            bm25     = build_bm25_index(passages)
            scratchpad = ''
            n_llm = n_slm = 0
            predicted = ''
            for step_idx in range(8):
                prompt  = build_react_prompt(question, scratchpad)
                slm_out = generate_single_with_logits(slm_m, slm_t, prompt)
                # Use full feature vector for learned router
                escalate = learned_router.decide_from_features(slm_out)
                if escalate:
                    llm_out = generate_single_with_logits(llm_m, llm_t, prompt)
                    resp = llm_out['response']; n_llm += 1
                else:
                    resp = slm_out['response']; n_slm += 1
                parsed  = parse_react_action(resp)
                is_term = _is_finish_action(parsed['action'], parsed['action_input'], resp)
                if is_term:
                    predicted = _extract_finish_answer(parsed['action'], parsed['action_input'], resp)
                    break
                obs = _execute_search(parsed['action_input'] or parsed['thought'], bm25, passages)
                scratchpad += f"\nThought: {parsed['thought']}\nAction: {parsed['action']}[{parsed['action_input']}]\nObservation: {obs}\n"
            success = _exact_match(predicted or '', gold)
            n_total = n_llm + n_slm
            all_rows.append({
                'question_id': sample['id'], 'dataset': ds_name,
                'policy': policy_id, 'slm_name': slm_cfg.name,
                'success': success, 'predicted': predicted, 'gold_answer': gold,
                'n_steps': step_idx+1, 'n_llm_calls': n_llm, 'n_slm_calls': n_slm,
                'llm_call_ratio': n_llm/n_total if n_total>0 else 0,
            })
        pd.DataFrame(all_rows).to_csv(LR_PATH, index=False)
        done_keys.add(key)
        sr = sum(r['success'] for r in all_rows if r['policy']==policy_id and r['dataset']==ds_name)
        n  = sum(1           for r in all_rows if r['policy']==policy_id and r['dataset']==ds_name)
        lr_ratio = sum(r['llm_call_ratio'] for r in all_rows if r['policy']==policy_id and r['dataset']==ds_name)/n
        print(f'  → SR={sr/n:.3f}, LLM_ratio={lr_ratio:.3f}')

del slm_m, slm_t, llm_m, llm_t
gc.collect(); torch.cuda.empty_cache()
print('\n✅ STEP 4 COMPLETE')

In [ ]:
from src.session_save import save
save(P['results'])
print('⚠️  Upload results before closing session')